## Imports

In [0]:
import requests
import zipfile
import os
from datetime import datetime

## Setting Path for Downloading Dataset

In [0]:
catalog, schema, volume = "bts_flight_data", "bronze", "bts_flight_dataset"
volume_path = f"/Volumes/{catalog}/{schema}/{volume}"
zip_dir = f"{volume_path}/zipped"
extract_dir = f"{volume_path}/unzipped"

os.makedirs(zip_dir, exist_ok=True)
os.makedirs(extract_dir, exist_ok=True)

START_YEAR = 2025
START_MONTH = 1

## Determine which months are already loaded

In [0]:
table_exists = spark.catalog.tableExists(f"{catalog}.{schema}.flights_raw")

if table_exists:
    existing_rows = spark.sql(f"""
        SELECT DISTINCT Year, Month FROM {catalog}.{schema}.flights_raw
    """).collect()
    already_loaded = {(row.Year, row.Month) for row in existing_rows}
else:
    already_loaded = set()

print(f"Already loaded months: {sorted(already_loaded)}")

## Build the list of months

In [0]:
now = datetime.now()
months_to_check = []

year, month = START_YEAR, START_MONTH
while (year, month) <= (now.year, now.month):
    months_to_check.append((year, month))
    month += 1
    if month > 12:
        month = 1
        year += 1

months_needed = [ym for ym in months_to_check if ym not in already_loaded]
print(f"Months to attempt downloading: {months_needed}")

## Running Loop to Download All Files for 2025

In [0]:
newly_downloaded = []

for year, month in months_needed:
    url = f"https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
    dest = f"{zip_dir}/flights_{year}_{month:02d}.zip"

    if os.path.exists(dest):
        print(f"Zip already present locally: {dest}")
        newly_downloaded.append((year, month, dest))
        continue

    resp = requests.get(url, verify=False, stream=True)

    if resp.status_code == 200:
        with open(dest, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
        print(f"Downloaded {year}-{month:02d}")
        newly_downloaded.append((year, month, dest))
    elif resp.status_code == 404:
        print(f"{year}-{month:02d} not yet published by BTS — skipping, will retry next run")
    else:
        print(f"{year}-{month:02d} failed unexpectedly: status {resp.status_code}")

print(f"\nNewly downloaded this run: {[(y, m) for y, m, _ in newly_downloaded]}")

## Unzipping Only new files

In [0]:
for year, month, zip_path in newly_downloaded:
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)
    print(f"Extracted {year}-{month:02d}")

## Reporting Size Before and After Unzipping

In [0]:
def get_folder_size(path):
    total_bytes = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_bytes += os.path.getsize(fp)
    return total_bytes

zip_size = get_folder_size(zip_dir)
extracted_size = get_folder_size(extract_dir)

print(f"\nZipped size:     {zip_size / (1024**3):.2f} GB")
print(f"Extracted size:  {extracted_size / (1024**3):.2f} GB")
if zip_size > 0:
    print(f"Expansion ratio: {extracted_size / zip_size:.1f}x")

## Load only the new month(s) and APPEND (not overwrite)

In [0]:
if not newly_downloaded:
    print("No new months to load — table is already up to date.")
else:
    # Build a file pattern matching only the newly extracted months' CSVs
    new_csv_patterns = [f"{extract_dir}/*{year}_{month}*.csv" for year, month, _ in newly_downloaded]

    df_new = spark.read.csv(new_csv_patterns, header=True, inferSchema=True)

    row_count = df_new.count()
    print(f"New rows to append: {row_count:,}")

    if table_exists:
        df_new.write.format("delta").mode("append").saveAsTable(f"{catalog}.{schema}.flights_raw")
        print("Appended to existing flights_raw table.")
    else:
        df_new.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.flights_raw")
        print("Created flights_raw table (first run).")